# Evaluación de Técnicas de Sobremuestreo con Hiperparámetros Fijos

## Diseño experimental

Este notebook implementa la comparación **justa** entre técnicas de sobremuestreo.

**Criterio adoptado:** El modelo Random Forest se configura con un único conjunto de
hiperparámetros fijos, aplicado de manera idéntica a **todas** las condiciones:
- Conjunto base (desbalanceado)
- Técnicas clásicas: SMOTE, Borderline SMOTE, ADASYN
- Técnicas contemporáneas: Radius-SMOTE, LD-SMOTE, VS-SMOTE
- PC-SMOTE (variantes)

Esto garantiza que las diferencias en F1-macro sean atribuibles **exclusivamente**
a la técnica de sobremuestreo, y no a una ventaja de optimización de hiperparámetros.


## Hiperparámetros fijos
| Parámetro        | Valor  |
|------------------|--------|
| n_estimators     | 300    |
| max_features     | sqrt   |
| criterion        | gini   |
| bootstrap        | True   |
| random_state     | 42     |

In [1]:
import sys
sys.path.append("../scripts")
sys.path.append("../datasets")

import os

# Rutas de datasets y resultados
RUTA_DATASETS_BASE = "../datasets/datasets_aumentados/base/"
RUTA_DATASETS_AUMENTADOS = "../datasets/datasets_aumentados/"
RUTA_DATASETS_CLASICOS = "../datasets/datasets_aumentados/resampler_clasicos/"
RUTA_DATASETS_CONTEMPORANEOS = "../datasets/datasets_aumentados/contemporaneos/"
DIRECTORIO_SALIDA = "../resultados"

os.makedirs(DIRECTORIO_SALIDA, exist_ok=True)
os.makedirs(RUTA_DATASETS_CLASICOS, exist_ok=True)
os.makedirs(RUTA_DATASETS_CONTEMPORANEOS, exist_ok=True)

In [2]:
import gc, time
from dataclasses import dataclass, asdict
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    recall_score,
    accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_STATE = 42

# =========================================================
# HIPERPARÁMETROS FIJOS (aplicados a TODOS los modelos)
# =========================================================
HIPERPARAMETROS_FIJOS = {
    "classifier__n_estimators" : 300,
    "classifier__max_features" : "sqrt",
    "classifier__criterion"    : "gini",
    "classifier__bootstrap"    : True,
}

print("✅ Hiperparámetros fijos cargados:")
for k, v in HIPERPARAMETROS_FIJOS.items():
    print(f"   {k}: {v}")

# Archivo Excel de salida
NOMBRE_ARCHIVO_EXCEL = os.path.join(DIRECTORIO_SALIDA, "resultados_params_fijos_vs_contemporaneos.xlsx")

✅ Hiperparámetros fijos cargados:
   classifier__n_estimators: 300
   classifier__max_features: sqrt
   classifier__criterion: gini
   classifier__bootstrap: True


In [3]:
from pathlib import Path
import re

@dataclass
class DatasetCombination:
    dataset_logico: str
    tipo_combination: str      # "base" | "clasico" | "contemporaneo" | "pcsmote"
    ruta_train_csv: str
    ruta_test_csv: str

    tecnica_aumento: str = "base"

    valor_densidad: str | int | None = "--"
    valor_riesgo: str | int | None = "--"
    criterio_pureza: str | None = "--"

    percentil_radio_distancia: str | int | None = "--"
    percentil_riesgo: str | int | None = "--"
    umbral_densidad: str | None = "--"
    umbral_riesgo: str | None = "--"

    tipo_pureza: str = "--"
    nombre_configuracion: str = ""

    grado_limpieza: str | int = "--"
    total_muestras_train: int | None = None
    tamanio_dataset: int | None = None
    sinteticos_generados: int = 0
    semillas_validas: int = 0
    semillas_analizadas: int = 0


@dataclass
class RegistroRendimiento:
    dataset_logico: str
    tipo_combination: str
    nombre_modelo_aprendizaje: str
    nombre_configuracion: str
    tecnica_aumento: str
    valor_densidad: str
    valor_riesgo: str
    criterio_pureza: str
    grado_limpieza: str

    cantidad_train: int
    cantidad_test: int
    cantidad_caracteristicas: int

    # Métricas Test
    test_f1_macro: float
    test_f1_weighted: float
    test_balanced_accuracy: float
    test_recall_macro: float
    test_accuracy: float

    hiperparametros_usados: str
    tiempo_entrenamiento_seg: float

In [4]:
def enumerar_combinaciones_base_y_aumentadas(
    ruta_base,
    ruta_clasicos,
    ruta_contemporaneos,
    ruta_aumentados,
    verbose=True
):
    combinaciones = []
    cont_combinaciones = 0
    tamanio_train_base_por_dataset_y_I = {}

    if verbose:
        print(f"[SCAN] Explorando carpeta base: {ruta_base}")
    archivos_base = os.listdir(ruta_base)

    def resolver_test(dataset_logico):
        patron_test = re.compile(rf"^{re.escape(dataset_logico)}_tdataset(\d+)_tm(\d+)_test\.csv$")
        nombre_test = None
        n_test_detectado = None
        tamanio_dataset = None

        for nc in archivos_base:
            m_test = patron_test.match(nc)
            if m_test:
                n_tm = int(m_test.group(2))
                if n_test_detectado is None or n_tm > n_test_detectado:
                    n_test_detectado = n_tm
                    nombre_test = nc
                    tamanio_dataset = int(m_test.group(1))

        return nombre_test, tamanio_dataset

    # 1) BASE
    for nombre in archivos_base:
        if not nombre.endswith("_train.csv"):
            continue

        m = re.match(r"(.+?)_I(\d+)_tm(\d+)_train\.csv$", nombre)
        if not m:
            continue

        dataset_logico = m.group(1)
        grado_limpieza = int(m.group(2))
        total_muestras_train = int(m.group(3))
        clave_base = (dataset_logico, grado_limpieza)
        tamanio_train_base_por_dataset_y_I[clave_base] = total_muestras_train

        nombre_test, tamanio_dataset = resolver_test(dataset_logico)
        if nombre_test is None:
            continue

        cont_combinaciones += 1
        if verbose:
            print(f"#{cont_combinaciones}  [OK] BASE: {nombre}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="base",
            ruta_train_csv=os.path.join(ruta_base, nombre),
            ruta_test_csv=os.path.join(ruta_base, nombre_test),
            tecnica_aumento="base",
            grado_limpieza=grado_limpieza,
            total_muestras_train=total_muestras_train,
            tamanio_dataset=tamanio_dataset,
        ))

    def agregar_combinaciones_simples(ruta_carpeta, tipo_combination, etiqueta):
        nonlocal cont_combinaciones

        if not os.path.isdir(ruta_carpeta):
            if verbose:
                print(f"[WARN] Carpeta inexistente: {ruta_carpeta}")
            return

        if verbose:
            print(f"[SCAN] Explorando carpeta {etiqueta}: {ruta_carpeta}")

        for nombre in os.listdir(ruta_carpeta):
            if not nombre.endswith("_train.csv"):
                continue

            m = re.match(r"(.+?)_(.+?)_I(\d+)_sg(\d+)_train\.csv$", nombre)
            if not m:
                continue

            tecnica = m.group(1).lower()
            dataset_logico = m.group(2)
            grado_limpieza = int(m.group(3))
            sinteticos_generados = int(m.group(4))

            clave_base = (dataset_logico, grado_limpieza)
            total_muestras_train = tamanio_train_base_por_dataset_y_I.get(clave_base)
            if total_muestras_train is None:
                continue

            nombre_test, tamanio_dataset = resolver_test(dataset_logico)
            if nombre_test is None:
                continue

            cont_combinaciones += 1
            if verbose:
                print(f"#{cont_combinaciones}  [OK] {etiqueta.upper()} ({tecnica}): {nombre}")

            combinaciones.append(DatasetCombination(
                dataset_logico=dataset_logico,
                tipo_combination=tipo_combination,
                ruta_train_csv=os.path.join(ruta_carpeta, nombre),
                ruta_test_csv=os.path.join(ruta_base, nombre_test),
                tecnica_aumento=tecnica,
                grado_limpieza=grado_limpieza,
                total_muestras_train=total_muestras_train,
                sinteticos_generados=sinteticos_generados,
                tamanio_dataset=tamanio_dataset,
            ))

    # 2) CLÁSICOS
    agregar_combinaciones_simples(
        ruta_carpeta=ruta_clasicos,
        tipo_combination="clasico",
        etiqueta="clásicos",
    )

    # 3) CONTEMPORÁNEOS
    agregar_combinaciones_simples(
        ruta_carpeta=ruta_contemporaneos,
        tipo_combination="contemporaneo",
        etiqueta="contemporáneos",
    )

    # 4) PC-SMOTE
    if verbose:
        print(f"[SCAN] Explorando carpeta aumentados: {ruta_aumentados}")
    archivos_aumentados = os.listdir(ruta_aumentados)
    patron_pcsmote = re.compile(
        r"^pcs_(?P<dataset>.+?)_"
        r"PRD(?P<prd>\d+)_"
        r"PR(?P<pr>\d+)_"
        r"CP(?P<cp>(?:ent|prop))_"
        r"UD(?P<ud>\d{3})_"
        r"(?P<tipo_pureza>(?:PE\d+|Upp\d{3}))_"
        r"UR(?P<ur>\d{3})_"
        r"I(?P<iso>\d+)_"
        r"SC(?P<sc>\d+)_"
        r"SV(?P<sv>\d+)_"
        r"SG(?P<sg>\d+)_train\.csv$"
    )

    for nombre in archivos_aumentados:
        if not nombre.endswith("_train.csv"):
            continue

        m = patron_pcsmote.match(nombre)
        if not m:
            continue

        dataset_logico = m.group("dataset")
        valor_densidad = int(m.group("prd"))
        valor_riesgo = int(m.group("pr"))
        cp_code = m.group("cp")
        ud_str = m.group("ud")
        ur_str = m.group("ur")
        tipo_pureza = m.group("tipo_pureza")
        grado_limpieza = int(m.group("iso"))
        semillas_analizadas = int(m.group("sc"))
        semillas_validas = int(m.group("sv"))
        sinteticos_generados = int(m.group("sg"))
        criterio_pureza = "entropia" if cp_code == "ent" else "proporcion"

        nombre_configuracion = (
            f"PRD{valor_densidad}_PR{valor_riesgo}_CP{cp_code}_"
            f"UD{ud_str}_UR{ur_str}_{tipo_pureza}_I{grado_limpieza}_"
            f"SC{semillas_analizadas}_SV{semillas_validas}_SG{sinteticos_generados}"
        )

        nombre_test, tamanio_dataset = resolver_test(dataset_logico)
        if nombre_test is None:
            continue

        cont_combinaciones += 1
        if verbose:
            print(f"#{cont_combinaciones}  [OK] PC-SMOTE: {nombre}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="pcsmote",
            ruta_train_csv=os.path.join(ruta_aumentados, nombre),
            ruta_test_csv=os.path.join(ruta_base, nombre_test),
            tecnica_aumento="pcsmote",
            valor_densidad=valor_densidad,
            valor_riesgo=valor_riesgo,
            criterio_pureza=criterio_pureza,
            percentil_radio_distancia=valor_densidad,
            percentil_riesgo=valor_riesgo,
            umbral_densidad=ud_str,
            umbral_riesgo=ur_str,
            grado_limpieza=grado_limpieza,
            sinteticos_generados=sinteticos_generados,
            semillas_validas=semillas_validas,
            semillas_analizadas=semillas_analizadas,
            tipo_pureza=tipo_pureza,
            nombre_configuracion=nombre_configuracion,
            tamanio_dataset=tamanio_dataset,
        ))

    if verbose:
        print(f"\n[INFO] Total combinaciones: {len(combinaciones)}")
    return combinaciones


print("[INFO] Enumerando combinaciones...")
combinaciones = enumerar_combinaciones_base_y_aumentadas(
    ruta_base=RUTA_DATASETS_BASE,
    ruta_clasicos=RUTA_DATASETS_CLASICOS,
    ruta_contemporaneos=RUTA_DATASETS_CONTEMPORANEOS,
    ruta_aumentados=RUTA_DATASETS_AUMENTADOS,
    verbose=True
)

[SCAN] Enumerando combinaciones...
[SCAN] Explorando carpeta base: ../datasets/datasets_aumentados/base/
#1  [OK] BASE: predict_faults_I0_tm8000_train.csv
#2  [OK] BASE: predict_faults_I1_tm7919_train.csv
#3  [OK] BASE: predict_faults_I3_tm7759_train.csv
#4  [OK] BASE: shuttle_I0_tm46400_train.csv
#5  [OK] BASE: shuttle_I1_tm45932_train.csv
#6  [OK] BASE: telco_churn_I0_tm5634_train.csv
#7  [OK] BASE: telco_churn_I1_tm5577_train.csv
#8  [OK] BASE: us_crime_I0_tm1595_train.csv
#9  [OK] BASE: us_crime_I1_tm1578_train.csv
#10  [OK] BASE: us_crime_I3_tm1546_train.csv
[SCAN] Explorando carpeta clásicos: ../datasets/datasets_aumentados/resampler_clasicos/
#11  [OK] CLÁSICOS (adasyn): adasyn_predict_faults_I0_sg7453_train.csv
#12  [OK] CLÁSICOS (adasyn): adasyn_predict_faults_I1_sg7380_train.csv
#13  [OK] CLÁSICOS (adasyn): adasyn_predict_faults_I3_sg7185_train.csv
#14  [OK] CLÁSICOS (adasyn): adasyn_shuttle_I0_sg208961_train.csv
#15  [OK] CLÁSICOS (adasyn): adasyn_shuttle_I1_sg206831_train.c

In [5]:
# Datasets a omitir del experimento (el resto se incluye)
EXCLUIR_DATASETS = {
    "iris", "glass", "heart", "wdbc", "ecoli",
    "gear_vibration",
}

tareas = [
    combo for combo in combinaciones
    if combo.dataset_logico.lower() not in EXCLUIR_DATASETS
]

print(f"📦 Total de tareas: {len(tareas)}")
print(f"   Datasets: {sorted(set(c.dataset_logico for c in tareas))}")

📦 Total de tareas: 102
   Datasets: ['predict_faults', 'shuttle', 'telco_churn', 'us_crime']


In [6]:
# =========================================================
# UTILIDADES
# =========================================================

def cargar_xy(ruta_csv):
    """Lee un CSV y devuelve (X, y). Usa 'target' si existe, si no la última columna."""
    df = pd.read_csv(ruta_csv)
    if "target" in df.columns:
        X = df.drop(columns=["target"]).to_numpy(dtype=np.float32, copy=False)
        y = df["target"].to_numpy()
    else:
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32, copy=False)
        y = df.iloc[:, -1].to_numpy()
    return X, y


def construir_pipeline_rf():
    """Instancia el Pipeline con RandomForest (sin configurar aún)."""
    return Pipeline([
        ("classifier", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
        ))
    ])


def entrenar_y_evaluar(pipeline, hiperparametros, X_train, y_train, X_test, y_test):
    """
    Aplica los hiperparámetros fijos, entrena y evalúa en test.
    Retorna dict con métricas y tiempo.
    """
    inicio = time.perf_counter()

    pipeline.set_params(**hiperparametros)
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    elapsed = time.perf_counter() - inicio

    return dict(
        tiempo=float(elapsed),
        f1_macro=float(f1_score(y_test, y_pred, average="macro")),
        f1_weighted=float(f1_score(y_test, y_pred, average="weighted")),
        balanced_accuracy=float(balanced_accuracy_score(y_test, y_pred)),
        recall_macro=float(recall_score(y_test, y_pred, average="macro")),
        accuracy=float(accuracy_score(y_test, y_pred)),
    )

In [7]:
# =========================================================
# EVALUACIÓN MASIVA — hiperparámetros fijos para todos
# =========================================================

registros = []
total = len(tareas)
inicio_total = time.perf_counter()

print("\n" + "="*100)
print("🏁 EVALUACIÓN CON HIPERPARÁMETROS FIJOS — todas las técnicas en igualdad de condiciones")
print(f"   n_estimators={HIPERPARAMETROS_FIJOS['classifier__n_estimators']} | "
      f"max_features={HIPERPARAMETROS_FIJOS['classifier__max_features']} | "
      f"criterion={HIPERPARAMETROS_FIJOS['classifier__criterion']}")
print("="*100)

for idx, combo in enumerate(tareas, start=1):

    print(f"\n{'='*80}")
    print(f"🏁 [{idx}/{total}] Dataset: {combo.dataset_logico} | "
          f"Tipo: {combo.tipo_combination} | Técnica: {combo.tecnica_aumento}")
    print(f"📂 Train: {os.path.basename(combo.ruta_train_csv)}")

    try:
        X_train, y_train = cargar_xy(combo.ruta_train_csv)
        X_test,  y_test  = cargar_xy(combo.ruta_test_csv)
    except Exception as e:
        print(f"❌ Error cargando CSV: {e}")
        continue

    pipeline = construir_pipeline_rf()

    try:
        res = entrenar_y_evaluar(
            pipeline=pipeline,
            hiperparametros=HIPERPARAMETROS_FIJOS,
            X_train=X_train, y_train=y_train,
            X_test=X_test,   y_test=y_test,
        )
    except Exception as e:
        print(f"❌ Error durante entrenamiento: {e}")
        continue

    print(f"✅ Completado en {res['tiempo']:.2f} s")
    print(f"📊 F1-macro(Test): {res['f1_macro']:.4f}  |  "
          f"Bal.Acc: {res['balanced_accuracy']:.4f}  |  "
          f"Accuracy: {res['accuracy']:.4f}")

    registros.append(asdict(RegistroRendimiento(
        dataset_logico=combo.dataset_logico,
        tipo_combination=combo.tipo_combination,
        nombre_modelo_aprendizaje="RandomForest",
        nombre_configuracion=combo.nombre_configuracion,
        tecnica_aumento=combo.tecnica_aumento,
        valor_densidad=str(combo.valor_densidad),
        valor_riesgo=str(combo.valor_riesgo),
        criterio_pureza=str(combo.criterio_pureza),
        grado_limpieza=str(combo.grado_limpieza),

        cantidad_train=int(X_train.shape[0]),
        cantidad_test=int(X_test.shape[0]),
        cantidad_caracteristicas=int(X_train.shape[1]),

        test_f1_macro=round(res["f1_macro"], 4),
        test_f1_weighted=round(res["f1_weighted"], 4),
        test_balanced_accuracy=round(res["balanced_accuracy"], 4),
        test_recall_macro=round(res["recall_macro"], 4),
        test_accuracy=round(res["accuracy"], 4),

        hiperparametros_usados=str(HIPERPARAMETROS_FIJOS),
        tiempo_entrenamiento_seg=round(res["tiempo"], 2),
    )))

    gc.collect()

elapsed_total = time.perf_counter() - inicio_total
print(f"\n⏱️ Tiempo total: {elapsed_total/60:.2f} min")
print(f"📦 Registros generados: {len(registros)}")


🏁 EVALUACIÓN CON HIPERPARÁMETROS FIJOS — todas las técnicas en igualdad de condiciones
   n_estimators=300 | max_features=sqrt | criterion=gini

🏁 [1/102] Dataset: predict_faults | Tipo: base | Técnica: base
📂 Train: predict_faults_I0_tm8000_train.csv
✅ Completado en 2.39 s
📊 F1-macro(Test): 0.8594  |  Bal.Acc: 0.8125  |  Accuracy: 0.9835

🏁 [2/102] Dataset: predict_faults | Tipo: base | Técnica: base
📂 Train: predict_faults_I1_tm7919_train.csv
✅ Completado en 2.18 s
📊 F1-macro(Test): 0.8509  |  Bal.Acc: 0.8051  |  Accuracy: 0.9825

🏁 [3/102] Dataset: predict_faults | Tipo: base | Técnica: base
📂 Train: predict_faults_I3_tm7759_train.csv
✅ Completado en 1.96 s
📊 F1-macro(Test): 0.8518  |  Bal.Acc: 0.8252  |  Accuracy: 0.9815

🏁 [4/102] Dataset: shuttle | Tipo: base | Técnica: base
📂 Train: shuttle_I0_tm46400_train.csv
✅ Completado en 5.95 s
📊 F1-macro(Test): 0.8549  |  Bal.Acc: 0.8571  |  Accuracy: 0.9997

🏁 [5/102] Dataset: shuttle | Tipo: base | Técnica: base
📂 Train: shuttle_I1_tm4

In [8]:
# =========================================================
# EXPORTAR RESULTADOS A EXCEL
# =========================================================

if not registros:
    print("❌ No hay registros para exportar.")
else:
    df = pd.DataFrame(registros)

    # Ordenar para lectura cómoda: dataset > tipo > técnica > grado
    df.sort_values(
        ["dataset_logico", "tipo_combination", "tecnica_aumento", "grado_limpieza"],
        inplace=True,
    )
    df.reset_index(drop=True, inplace=True)

    df.to_excel(NOMBRE_ARCHIVO_EXCEL, index=False)
    print(f"✅ Resultados guardados en: {NOMBRE_ARCHIVO_EXCEL}")
    print(f"   Filas: {len(df)} | Columnas: {len(df.columns)}")
    print("\n📋 Vista previa:")
    display(df[["dataset_logico", "tipo_combination", "tecnica_aumento",
                "grado_limpieza", "cantidad_train",
                "test_f1_macro", "test_balanced_accuracy"]].head(20))

✅ Resultados guardados en: ../resultados\resultados_params_fijos_vs_contemporaneos.xlsx
   Filas: 102 | Columnas: 19

📋 Vista previa:


,dataset_logico,tipo_combination,tecnica_aumento,grado_limpieza,cantidad_train,test_f1_macro,test_balanced_accuracy
0,predict_faults,base,base,0,8000,0.8594,0.8125
1,predict_faults,base,base,1,7919,0.8509,0.8051
2,predict_faults,base,base,3,7759,0.8518,0.8252
3,predict_faults,clasico,adasyn,0,15453,0.8083,0.8274
4,predict_faults,clasico,adasyn,1,15299,0.8154,0.8415
5,predict_faults,clasico,adasyn,3,14944,0.8132,0.8478
6,predict_faults,clasico,borderlinesmote,0,15444,0.8174,0.8152
7,predict_faults,clasico,borderlinesmote,1,15288,0.8274,0.8229
8,predict_faults,clasico,borderlinesmote,3,14980,0.8272,0.8295
9,predict_faults,clasico,smote,0,15444,0.8081,0.8208


In [9]:
# =========================================================
# TABLA COMPARATIVA: baseline vs técnicas de sobremuestreo
# =========================================================

if registros:
    df_cmp = pd.DataFrame(registros)

    # F1-macro del baseline por (dataset, grado_limpieza)
    base_f1 = (
        df_cmp[df_cmp["tipo_combination"] == "base"]
        .set_index(["dataset_logico", "grado_limpieza"])["test_f1_macro"]
    )

    def delta_vs_base(row):
        key = (row["dataset_logico"], row["grado_limpieza"])
        if key in base_f1.index:
            return round(row["test_f1_macro"] - base_f1[key], 4)
        return float("nan")

    df_cmp["delta_vs_base"] = df_cmp.apply(delta_vs_base, axis=1)

    print("\n[INFO] Comparativa F1-macro (Test) - hiperparámetros fijos para todos:")
    print(df_cmp[
        ["dataset_logico", "tipo_combination", "tecnica_aumento", "grado_limpieza",
         "cantidad_train", "test_f1_macro", "delta_vs_base"]
    ].sort_values(
        ["dataset_logico", "grado_limpieza", "tipo_combination", "test_f1_macro"],
        ascending=[True, True, True, False]
    ).to_string(index=False))


[INFO] Comparativa F1-macro (Test) - hiperparámetros fijos para todos:
dataset_logico tipo_combination tecnica_aumento grado_limpieza  cantidad_train  test_f1_macro  delta_vs_base
predict_faults             base            base              0            8000         0.8594         0.0000
predict_faults          clasico borderlinesmote              0           15444         0.8174        -0.0420
predict_faults          clasico          adasyn              0           15453         0.8083        -0.0511
predict_faults          clasico           smote              0           15444         0.8081        -0.0513
predict_faults    contemporaneo        vs-smote              0           15444         0.8328        -0.0266
predict_faults    contemporaneo    radius-smote              0           15444         0.8124        -0.0470
predict_faults    contemporaneo        ld-smote              0           15444         0.8063        -0.0531
predict_faults          pcsmote         pcsmote         